# ViewSet

In [ ]:
# Serializer para Usuario
# src\users\serializers.py
from rest_framework import serializers
GENDER = [
    ("female","Femenino"),
    ("male","Masculino"),
    ("other","Otro")
]

class UserSerializer(serializers.Serializer):
    name            = serializers.CharField(max_length=100)
    paternal_name   = serializers.CharField(max_length=100)
    maternal_name   = serializers.CharField(max_length=100)
    age             = serializers.IntegerField()
    email           = serializers.EmailField()
    gender          = serializers.ChoiceField(choices=GENDER)
    phone           = serializers.CharField(max_length=20)

In [ ]:
# Asignación para ViewSet
# src\users\views.py
from django.shortcuts import render
from .forms import UserForm

from rest_framework.response import Response
from rest_framework import status
from rest_framework.viewsets import ViewSet
from .serializers import UserSerializer
from .models import User



class APIUser(ViewSet):
    user_serializer = UserSerializer

    def list(self, request):
        """Listado de usuarios registrados"""
        users = User.objects.all()
        serializer = self.user_serializer(users, many=True)
        return Response({
            "list_users": serializer.data
        })
    
    def create(self, request):
        """Crear un nuevo usuario"""
        serializer = self.user_serializer(data=request.data)
        if serializer.is_valid():
            user = User.objects.create(**serializer.validated_data)
            user_data = self.user_serializer(user).data
            return Response({
                "message": f"Usuario creado con ID {user.id}",
                "data": user_data
            }, status=status.HTTP_201_CREATED)
        return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

    def retrieve(self, request, pk=None):
        """Consulta de un usuario por su id"""
        try:
            user = User.objects.get(pk=pk)
        except User.DoesNotExist:
            return Response({"error": "Usuario no encontrado"}, status=status.HTTP_404_NOT_FOUND)
        serializer = self.user_serializer(user)
        return Response(serializer.data)

    def update(self, request, pk=None):
        """Actualización completa de un usuario"""
        try:
            user = User.objects.get(pk=pk)
        except User.DoesNotExist:
            return Response({"error": "Usuario no encontrado"}, status=status.HTTP_404_NOT_FOUND)

        serializer = self.user_serializer(user, data=request.data)
        if serializer.is_valid():
            for attr, value in serializer.validated_data.items():
                setattr(user, attr, value)
            user.save()
            return Response(self.user_serializer(user).data)
        return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

    def partial_update(self, request, pk=None):
        """Actualización parcial de un usuario"""
        try:
            user = User.objects.get(pk=pk)
        except User.DoesNotExist:
            return Response({"error": "Usuario no encontrado"}, status=status.HTTP_404_NOT_FOUND)

        serializer = self.user_serializer(user, data=request.data, partial=True)
        if serializer.is_valid():
            for attr, value in serializer.validated_data.items():
                setattr(user, attr, value)
            user.save()
            return Response(self.user_serializer(user).data)
        return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

    def destroy(self, request, pk=None):
        """Eliminación de un usuario"""
        try:
            user = User.objects.get(pk=pk)
        except User.DoesNotExist:
            return Response({"error": "Usuario no encontrado"}, status=status.HTTP_404_NOT_FOUND)
        user.delete()
        return Response({"message": f"Usuario con ID {pk} eliminado"}, status=status.HTTP_204_NO_CONTENT)

In [ ]:
# URLS
# src\users\urls.py
from django.urls import path, include
from .views import userCreate, APIUser
from rest_framework.routers import DefaultRouter

router = DefaultRouter()
router.register("api-user",APIUser, basename="api-user")



urlpatterns = [
    path("create/",userCreate),
    path("",include(router.urls)),
]
